# Experimentinng with the netcdf stitching

Using the recepies that were set up in the R devlopment.

Import the packages that are needed for this script.

In [29]:
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import fsspec
import pandas as pd

Import csv files created in R.

In [16]:
# import the objects
recepie = pd.read_csv('/Users/dorh012/Documents/2021/stitches/notebooks/stitches_dev/recpies_for_python.csv')
file_list = pd.read_csv('/Users/dorh012/Documents/2021/stitches/notebooks/stitches_dev/file_list.csv')

recepie = recepie[recepie["stitching_id"] == 1].copy()
file_list = recepie[recepie["stitching_id"] == 1].file.unique().copy()

Data has loaded


In [ ]:
cols = ['target_start_yr', 'target_end_yr', 'archive_start_yr', 'archive_end_yr', 'file']
recepie = recepie[cols].drop_duplicates().copy()

Define the function that reads in data from the pangeo cloud.


In [17]:
def load_data(path):
    """Extract data for a single file."""
    ds = xr.open_zarr(fsspec.get_mapper(path))
    return ds

# make sure that the funtion actually works!
load_data(file_list[0])


<xarray.Dataset>
Dimensions:    (bnds: 2, lat: 64, lon: 128, time: 1980)
Coordinates:
    height     float64 ...
  * lat        (lat) float64 -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
    lat_bnds   (lat, bnds) float64 dask.array<chunksize=(64, 2), meta=np.ndarray>
  * lon        (lon) float64 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
    lon_bnds   (lon, bnds) float64 dask.array<chunksize=(128, 2), meta=np.ndarray>
  * time       (time) object 1850-01-16 12:00:00 ... 2014-12-16 12:00:00
    time_bnds  (time, bnds) object dask.array<chunksize=(1980, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    tas        (time, lat, lon) float32 dask.array<chunksize=(600, 64, 128), meta=np.ndarray>
Attributes: (12/54)
    CCCma_model_hash:            f0c7ce6721e37102f4cbfb4d931d60ffc953a1b1
    CCCma_parent_runid:          rc3.1-pictrl
    CCCma_pycmor_hash:           33c30511acc319a98240633965a04ca99c26427e
    CCCma_runid:                 rc3.1-his24
    Conventions:                 CF-1.7 CMIP-6.2
    YMDH_branch_time_in_child:   1850:01:01:00
    ...                          ...
    title:                       CanESM5 output prepared for CMIP6
    tracking_id:                 hdl:21.14100/b988e4c3-d23a-4c8d-a1c5-e6f3172...
    variable_id:                 tas
    variant_label:               r24i1p1f1
    version:                     v20190429
    status:                      2019-10-25;created;by nhn2@columbia.edu

In [9]:
# Make an empty list
names_list = []
data_list = []

# this for loop is really really freaking slow need to figure out how
# the heck to do this also only want to have to run it once runing it mulitple
# times is awuful!
for f in file_list:
    names_list.append(f)
    o = load_data(f)
    data_list.append(o)

In [18]:
len(data_list)

48

In [21]:
# TODO make the function work with any variable
def get_netcdf_values(i, dl, rp, fl):
    """Extract data for a single file."""
    file = rp["file"][i]
    start_yr = str(rp["archive_start_yr"][i])
    end_yr = str(rp["archive_end_yr"][i])

    # Figure out which index level we are on and then get the
    # xarray from the list.
    index = int(np.where(fl == file)[0])
    extracted = dl[index]
    dat = extracted.sel(time=slice(start_yr, end_yr)).tas.values.copy()
    return dat

In [27]:
recepie.reset_index(drop=True, inplace=True)

out = get_netcdf_values(i=0, dl=data_list, rp=recepie, fl=file_list)

for i in range(1,len(recepie)):
    new_vals = get_netcdf_values(i=i, dl=data_list, rp=recepie, fl=file_list)
    out = np.concatenate((out, new_vals), axis = 0)

NameError: name 'single_recepie' is not defined

In [ ]:
# Create a time series with the target data information
# TODO this needs to be some sort of check that allows you to
# select monthly, daily, or annual data.
start = str(min(recepie["target_start_yr"]))
end = str(max(recepie["target_end_yr"]))
# I assume that there is some issues defining the time element here
# it might have been the leap year, or during the historical -> future
# scn.
times = pd.date_range(start = start, end=end, freq='M')
times = times[0:2988].copy()

lat = data_list[0].lat.values
lon = data_list[0].lon.values

In [ ]:
# okay so there is some problem with the times
rslt = xr.Dataset({'tas':xr.DataArray(
    out,
    coords=[times, lat, lon],
    dims=["time", "lat", 'lon'],
    attrs  = {'units' : 'K'})
})

rslt.to_netcdf("stiched1.nc")